# March 18 Periodicity Pregate - Gate Only

This is a lightweight variant of `march18_periodicity_pregate.ipynb` for testing only the CE-only pre-periodicity gate.

It does **not** run the expensive periodic branch (`python -m malca.events --baseline-func phase_template`) and does **not** run post-filter periodicity validation.


In [1]:
from pathlib import Path
import hashlib
import importlib
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root (missing pyproject.toml).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from malca.notebook_paths import infer_run_dir, localize_lightcurve_frame_paths, resolve_repo_path
from malca.config.config_pipeline import MAG_BINS as ALL_MAG_BINS, WORKERS as DEFAULT_WORKERS
import malca.periodicity_gate as malca_periodicity_gate
from malca.manifest import build_manifest

malca_periodicity_gate = importlib.reload(malca_periodicity_gate)
apply_pre_periodicity_gate = malca_periodicity_gate.apply_pre_periodicity_gate

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)

print(f"Repo root: {REPO_ROOT}")
print("Loaded local MALCA pregate implementation.")


/home/calder/miniforge3/envs/malca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: /home/calder/code/malca
Loaded local MALCA pregate implementation.


In [2]:
# Point INPUT_TABLE at an existing manifest/tag parquet with dat_path,
# or leave it as None and build a flat-directory manifest from FLAT_LC_DIR.
INPUT_TABLE = None
RUN_NAME = "runs_march18_bundle_all"
RUN_DIR = REPO_ROOT / "output" / "runs" / RUN_NAME
FLAT_LC_DIR = RUN_DIR / "bundle_assets" / "lightcurves"
INDEX_FILE = None
MAG_BINS = list(ALL_MAG_BINS)
N_WORKERS = min(8, DEFAULT_WORKERS)

# Set to a small integer, e.g. 200, for a quick smoke test before running all rows.
MAX_CANDIDATES = None

INPUT_TABLE = resolve_repo_path(INPUT_TABLE, repo_root=REPO_ROOT)
FLAT_LC_DIR = resolve_repo_path(FLAT_LC_DIR, repo_root=REPO_ROOT)
INDEX_FILE = resolve_repo_path(INDEX_FILE, repo_root=REPO_ROOT)

OUTPUT_DIR = REPO_ROOT / "output" / "diagnostics" / "march18_periodicity_pregate_gate_only"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The pregate is CE-only: period grid -> {P/2, P, 2P} -> folded scatter choice.
GATE_KWARGS = {
    "min_period": 0.2,
    "max_period": 100.0,
    "n_periods": 5000,
    "ce_snr_threshold": 10.0,
    "min_points": 50,
    "scatter_ratio_max": 0.9,
    "workers": N_WORKERS,
}

GATE_LOGIC_TAG = "ce_only_folded_scatter_v2_gate_only"
GATE_RUN_TAG = hashlib.md5(
    json.dumps(
        {"logic_tag": GATE_LOGIC_TAG, "max_candidates": MAX_CANDIDATES, **GATE_KWARGS},
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:10]

print(f"Using gate run tag: {GATE_RUN_TAG}")
print(f"Output dir: {OUTPUT_DIR}")


Using gate run tag: 5709196146
Output dir: /home/calder/code/malca/output/diagnostics/march18_periodicity_pregate_gate_only


In [3]:
if INPUT_TABLE is not None:
    input_path = Path(INPUT_TABLE)
    if input_path.suffix.lower() in {".parquet", ".pq"}:
        df_input = pd.read_parquet(input_path)
    else:
        df_input = pd.read_csv(input_path)

    input_run_dir = infer_run_dir(input_path) or infer_run_dir(RUN_DIR)
    df_input, localized_counts = localize_lightcurve_frame_paths(
        df_input,
        run_dir=input_run_dir,
        repo_root=REPO_ROOT,
        path_columns=("dat_path", "path", "lc_path"),
    )
    if localized_counts:
        print(f"Localized stored light-curve paths: {localized_counts}")

    input_path_col = "dat_path" if "dat_path" in df_input.columns else ("path" if "path" in df_input.columns else None)
    if input_path_col is None:
        raise KeyError("Input table must include a dat_path or path column.")

    exists_mask = df_input[input_path_col].map(lambda x: Path(str(x)).expanduser().exists() if pd.notna(x) else False)
    if not bool(exists_mask.all()):
        print(f"Retained {int(exists_mask.sum()):,} rows with local light curves.")
    df_input = df_input[exists_mask].reset_index(drop=True)
else:
    df_input = build_manifest(
        None,
        None,
        mag_bins=MAG_BINS,
        id_column="asas_sn_id",
        flat_lc_dir=FLAT_LC_DIR,
        index_file=INDEX_FILE,
        show_progress=True,
        n_workers=N_WORKERS,
    )
    df_input = df_input[df_input["dat_exists"]].reset_index(drop=True)

if MAX_CANDIDATES is not None:
    df_input = df_input.head(int(MAX_CANDIDATES)).reset_index(drop=True)
    print(f"Limited to first {len(df_input):,} candidates for smoke testing.")

print(f"Loaded {len(df_input):,} candidates")
display(df_input.head())


flat light curves: 100%|██████████| 9569/9569 [00:00<00:00, 346711.26it/s]

[warn] 9569 flat light curves lacked mag_bin metadata; downstream mag-bin filters will treat them as unparsed.
Loaded 9,569 candidates


,source_id,mag_bin,index_num,index_csv,lc_dir,lc_dir_exists,dat_path,dat_exists
0,103079222309,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True
1,103079223768,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True
2,103079224553,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True
3,103079238346,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True
4,103079246967,None,None,None,/home/calder/code/malca/output/runs/runs_march...,True,/home/calder/code/malca/output/runs/runs_march...,True


In [ ]:
path_col = "dat_path" if "dat_path" in df_input.columns else "path"
checkpoint_path = OUTPUT_DIR / f"march18_periodicity_gate_only_checkpoint_{GATE_RUN_TAG}.parquet"
output_file = OUTPUT_DIR / f"march18_periodicity_gate_only_{GATE_RUN_TAG}.parquet"

df_gate = apply_pre_periodicity_gate(
    df_input,
    path_col=path_col,
    checkpoint_path=checkpoint_path,
    show_tqdm=True,
    **GATE_KWARGS,
)
df_gate.to_parquet(output_file, index=False)
print(f"Saved gate output to {output_file}")
print(f"Checkpoint: {checkpoint_path}")

label_counts = df_gate["pre_periodicity_label"].value_counts(dropna=False).rename_axis("label").to_frame("n")
display(label_counts)

summary_cols = [
    "pre_periodicity_score",
    "pre_periodicity_scatter_ratio",
    "pre_ce_snr",
    "pre_ce_entropy",
]
summary_cols = [col for col in summary_cols if col in df_gate.columns]
display(df_gate.groupby("pre_periodicity_label")[summary_cols].median(numeric_only=True))


[pre_periodicity_gate] 0 cached, processing 9569 light curves


Pre-periodicity gate:   3%|▎         | 251/9569 [00:11<06:47, 22.87it/s]

In [ ]:
if df_gate.empty:
    print("No gate rows available for diagnostics.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    label_colors = {
        "periodic": "tab:green",
        "non_periodic": "tab:red",
    }

    threshold_ce_snr = float(GATE_KWARGS["ce_snr_threshold"])
    threshold_scatter = float(GATE_KWARGS["scatter_ratio_max"])

    df_gate["pre_periodicity_label"].value_counts().plot.bar(ax=axes[0, 0], title="Gate Labels")
    axes[0, 0].set_ylabel("count")

    df_gate["pre_periodicity_score"].dropna().plot.hist(ax=axes[0, 1], bins=40, title="Selected CE SNR")
    axes[0, 1].set_xlabel("pre_periodicity_score")
    axes[0, 1].axvline(threshold_ce_snr, color="0.35", ls="--", lw=1)

    df_gate["pre_periodicity_scatter_ratio"].dropna().plot.hist(ax=axes[1, 0], bins=40, title="Folded Scatter Ratio")
    axes[1, 0].set_xlabel("folded/raw scatter")
    axes[1, 0].axvline(threshold_scatter, color="0.35", ls="--", lw=1)

    for label, color in label_colors.items():
        subset = df_gate[df_gate["pre_periodicity_label"] == label]
        if subset.empty:
            continue
        axes[1, 1].scatter(
            subset["pre_ce_snr"],
            subset["pre_periodicity_scatter_ratio"],
            s=8,
            alpha=0.5,
            color=color,
            label=label,
        )
    axes[1, 1].axvline(threshold_ce_snr, color="0.35", ls="--", lw=1)
    axes[1, 1].axhline(threshold_scatter, color="0.35", ls="--", lw=1)
    axes[1, 1].set_title("CE SNR vs Folded Scatter")
    axes[1, 1].set_xlabel("pre_ce_snr")
    axes[1, 1].set_ylabel("pre_periodicity_scatter_ratio")
    axes[1, 1].legend(frameon=False)
    axes[1, 1].grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


In [ ]:
cols = [
    "source_id",
    "mag_bin",
    "pre_periodicity_label",
    "pre_periodicity_method",
    "pre_periodicity_base_period",
    "pre_periodicity_selected_period",
    "pre_periodicity_harmonic_factor",
    "pre_periodicity_score",
    "pre_periodicity_scatter_ratio",
    "pre_periodicity_support_count",
    "pre_ce_snr",
    "pre_ce_entropy",
    "pre_periodicity_reason",
]
cols = [col for col in cols if col in df_gate.columns]

print("Top confident periodic candidates")
display(df_gate[df_gate["pre_periodic_flag"]].sort_values("pre_periodicity_score", ascending=False)[cols].head(50))

print("Top non-periodic near-misses")
display(
    df_gate[~df_gate["pre_periodic_flag"]]
    .sort_values(["pre_periodicity_score", "pre_periodicity_scatter_ratio"], ascending=[False, True], na_position="last")[cols]
    .head(50)
)
